# «Вдвоём»: дообучение модели помощника

Нужно: **Среда выполнения → Сменить среду → GPU T4**, затем **Среда выполнения → Выполнить все**. Около 30–40 минут.

В конце адаптер (≈60 МБ) сам загрузится в GitHub (релиз `lora`), если в **Секретах** Colab (значок ключа слева) есть `GH_TOKEN` — токен GitHub с правом *Contents: Read and write* на репозиторий `vdvoem`. Без токена файл можно скачать из панели файлов слева и приложить к релизу `lora` вручную.

Дальше сервер сам скачает адаптер, проверит модель на запросах, которых она не видела, и включит её, только если она лучше текущей.

In [ ]:
BASE = "unsloth/Qwen2.5-3B-Instruct"   # совпадает с qwen2.5:3b на сервере
HF_BASE = "Qwen/Qwen2.5-3B-Instruct"
REPO = "vsershov21-spec/vdvoem"
!pip install -q unsloth
!git clone -q --depth 1 https://github.com/$REPO repo
!python repo/deploy/train/gen.py data

In [ ]:
import json, torch
from unsloth import FastLanguageModel
from unsloth.chat_templates import train_on_responses_only
from datasets import Dataset
from trl import SFTTrainer, SFTConfig

model, tok = FastLanguageModel.from_pretrained(BASE, max_seq_length=2048, load_in_4bit=True)
model = FastLanguageModel.get_peft_model(model, r=16, lora_alpha=32, lora_dropout=0, use_gradient_checkpointing="unsloth",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"])
load = lambda p: [json.loads(l) for l in open(p, encoding="utf-8") if l.strip()]
train, valid, test = load("data/train.jsonl"), load("data/valid.jsonl"), load("data/test.jsonl")
fmt = lambda r: {"text": tok.apply_chat_template(r["messages"], tokenize=False)}
trainer = SFTTrainer(model=model, tokenizer=tok, train_dataset=Dataset.from_list(train).map(fmt), eval_dataset=Dataset.from_list(valid).map(fmt),
    args=SFTConfig(dataset_text_field="text", per_device_train_batch_size=8, gradient_accumulation_steps=2, num_train_epochs=2,
                   learning_rate=2e-4, warmup_ratio=0.05, lr_scheduler_type="cosine", logging_steps=20, eval_strategy="epoch",
                   output_dir="out", report_to="none"))
trainer = train_on_responses_only(trainer, instruction_part="<|im_start|>user\n", response_part="<|im_start|>assistant\n")
trainer.train()

In [ ]:
# проверка на test — фразы и названия, которых не было в обучении
FastLanguageModel.for_inference(model)
def ask(msgs):
    ids = tok.apply_chat_template(msgs[:2], add_generation_prompt=True, return_tensors="pt").to("cuda")
    out = model.generate(ids, max_new_tokens=120, do_sample=False)
    return tok.decode(out[0][ids.shape[1]:], skip_special_tokens=True).strip()
ok_i = ok_all = 0
bad = []
for r in test:
    want = json.loads(r["messages"][2]["content"])
    try: got = json.loads(ask(r["messages"]))
    except Exception: got = {}
    ok_i += got.get("i") == want["i"]
    ok_all += got == want
    if got != want and len(bad) < 15: bad.append((r["messages"][1]["content"], want, got))
print(f"тип запроса верно: {ok_i}/{len(test)} · полностью верно: {ok_all}/{len(test)}")
for b in bad: print(" ✗", b)

In [ ]:
# адаптер → GGUF (для Ollama: FROM qwen2.5:3b + ADAPTER)
model.save_pretrained("lora"); tok.save_pretrained("lora")
!git clone -q --depth 1 https://github.com/ggml-org/llama.cpp
!pip install -q -r llama.cpp/requirements/requirements-convert_lora_to_gguf.txt || pip install -q gguf sentencepiece safetensors
!python llama.cpp/convert_lora_to_gguf.py lora --base-model-id $HF_BASE --outtype f16 --outfile vdvoem-intent-lora.gguf
!ls -la vdvoem-intent-lora.gguf

In [ ]:
# загрузка в GitHub, релиз «lora» (нужен секрет GH_TOKEN)
import requests, os
try:
    from google.colab import userdata
    token = userdata.get("GH_TOKEN")
except Exception:
    token = None
if not token:
    print("Нет GH_TOKEN — скачай vdvoem-intent-lora.gguf из панели файлов и приложи к релизу lora на GitHub.")
else:
    H = {"Authorization": f"Bearer {token}", "Accept": "application/vnd.github+json"}
    api = f"https://api.github.com/repos/{REPO}/releases"
    r = requests.get(f"{api}/tags/lora", headers=H)
    rel = r.json() if r.ok else requests.post(api, headers=H, json={"tag_name": "lora", "name": "Дообученная модель помощника",
        "body": "Адаптер LoRA к qwen2.5:3b. Ставится на сервер deploy/train/install-lora.sh", "prerelease": True}).json()
    for a in rel.get("assets", []):
        if a["name"] == "vdvoem-intent-lora.gguf":
            requests.delete(a["url"], headers=H)
    up = rel["upload_url"].split("{")[0] + "?name=vdvoem-intent-lora.gguf"
    with open("vdvoem-intent-lora.gguf", "rb") as f:
        res = requests.post(up, headers={**H, "Content-Type": "application/octet-stream"}, data=f)
    print("загружено" if res.ok else res.text)